***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [6. 成像中的去卷积](6_0_introduction.ipynb)
    * 上一节： [6.3 CLEAN 的实现方式](6_3_clean_flavours.ipynb)
    * 下一节： [6.5 源搜索与检测](6_5_source_finding.ipynb)

***


导入标准模块:


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

plt.rcParams['figure.max_open_warning'] = 0

NOTEBOOK_DIR = Path('6_Deconvolution') if Path('6_Deconvolution').exists() else Path('.')
NOTEBOOK_DIR = NOTEBOOK_DIR.resolve()
STYLE_PATH = NOTEBOOK_DIR.parent / 'style' / 'course.css'
TOGGLE_PATH = NOTEBOOK_DIR.parent / 'style' / 'code_toggle.html'

try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None

if STYLE_PATH.exists():
    try:
        HTML(STYLE_PATH.read_text(encoding='utf-8'))
    except Exception:
        pass


本节以下示例全部使用 notebook 内生成的合成观测，因此不再依赖外部 FITS 文件。这样做的目的是把“残图诊断、动态范围、停止准则”这些核心概念拆开来讲：我们已知真实天空、已知脏 PSF，也能明确地区分欠清理、接近最佳停止点和过清理三种状态。


In [ ]:
if TOGGLE_PATH.exists():
    try:
        HTML(TOGGLE_PATH.read_text(encoding='utf-8'))
    except Exception:
        pass


***


## 6.4 残图与图像质量<a id='deconv:sec:iqa'></a>


使用 `CLEAN` 或其他去卷积方法，通常都能得到比脏图“更好看”的图像，当然前提是不至于把去卷积做得失控。但问题在于：什么才叫“更好”的图像？这个问题其实几乎从来没有被严格定义过。

在综合成像实践里，人们讨论图像质量时，很少真正依赖单一的定量指标，反而更多依赖观察者的主观经验。坦率地说，这并不算十分科学。随着自动校准、自动成像和自动去卷积管线越来越普及，射电干涉测量迟早也需要一套更系统的图像质量评价框架。

因此，当我们从可见度数据生成最终图像时，实际上面临两个彼此相关的问题：

* 什么时候应该停止去卷积？
* 什么样的图像才算“好图”？

在 [$\S$ 6.2](6_2_clean.ipynb) 中，我们已经讨论过怎样利用迭代 `CLEAN` 从噪声中分离出天空模型，但并没有真正回答“何时停止”的问题。这个停止点本身没有统一定义。实践中最常见的做法，是给定一个迭代次数、一个阈值，或者一个经验性的残差水平，然后不断调整参数，直到得到一幅“看起来不错”的图像。

这一节我们专门讨论图像质量与残图诊断。为了避免真实观测中的校准误差把问题搅在一起，下面统一采用一组合成观测，并用同一个脏波束分别构造欠清理、接近最佳停止点和过清理三种结果。这样既能直接计算某些“真实误差”，也能看清哪些指标在真实数据上是可用的，哪些只是模拟里才有的参考量。


In [ ]:
def generalGauss2d(x0, y0, sigmax, sigmay, amp=1.0, theta=0.0):
    rtheta = np.deg2rad(theta)
    a = (np.cos(rtheta) ** 2.0) / (2.0 * sigmax ** 2.0) + (np.sin(rtheta) ** 2.0) / (2.0 * sigmay ** 2.0)
    b = -(np.sin(2.0 * rtheta)) / (4.0 * sigmax ** 2.0) + (np.sin(2.0 * rtheta)) / (4.0 * sigmay ** 2.0)
    c = (np.sin(rtheta) ** 2.0) / (2.0 * sigmax ** 2.0) + (np.cos(rtheta) ** 2.0) / (2.0 * sigmay ** 2.0)
    return lambda x, y: amp * np.exp(-1.0 * (a * (x - x0) ** 2.0 - 2.0 * b * (x - x0) * (y - y0) + c * (y - y0) ** 2.0))


def centered_gaussian(n, fwhm_x, fwhm_y=None, theta=0.0, amp=1.0):
    if fwhm_y is None:
        fwhm_y = fwhm_x
    sigma_x = fwhm_x / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    sigma_y = fwhm_y / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    yy, xx = np.mgrid[0:n, 0:n].astype(float)
    arr = generalGauss2d((n - 1) / 2.0, (n - 1) / 2.0, sigma_x, sigma_y, amp=amp, theta=theta)(xx, yy)
    return arr / arr.max()


def fft_convolve_same(image, kernel):
    return np.real(
        np.fft.fftshift(
            np.fft.ifft2(
                np.fft.fft2(np.fft.ifftshift(image)) * np.fft.fft2(np.fft.ifftshift(kernel))
            )
        )
    )


def shift_image(image, dy, dx):
    return np.roll(np.roll(image, int(dy), axis=0), int(dx), axis=1)


def make_uv_dirty_beam(n, n_pairs=240, radius_frac=0.30, seed=13):
    rng = np.random.default_rng(seed)
    center = n // 2
    radius = radius_frac * n
    angles = rng.uniform(0.0, 2.0 * np.pi, n_pairs)
    radii = radius * np.sqrt(rng.uniform(0.02, 1.0, n_pairs))
    u = np.rint(radii * np.cos(angles)).astype(int)
    v = np.rint(radii * np.sin(angles)).astype(int)
    keep = (np.abs(u) < n // 2 - 2) & (np.abs(v) < n // 2 - 2) & ((u != 0) | (v != 0))
    half = np.column_stack([u[keep], v[keep]])
    coords = np.vstack([half, -half])
    sampling = np.zeros((n, n), dtype=float)
    sampling[coords[:, 1] + center, coords[:, 0] + center] = 1.0
    psf = np.real(np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(sampling))))
    psf /= psf.max()
    return sampling, psf, coords


def hogbom_clean(dirty, psf, gain=0.18, niter=2000, threshold=0.0, clean_beam=None, capture_iters=(0,)):
    capture_iters = set(capture_iters)
    residual = dirty.copy()
    model = np.zeros_like(dirty)
    center = np.array(psf.shape) // 2
    states = {}

    def capture(iteration):
        model_image = fft_convolve_same(model, clean_beam) if clean_beam is not None else model.copy()
        restored = model_image + residual
        states[iteration] = {
            'model': model.copy(),
            'residual': residual.copy(),
            'model_image': model_image.copy(),
            'restored': restored.copy(),
        }

    capture(0)
    last_iter = 0
    for iteration in range(1, niter + 1):
        iy, ix = np.unravel_index(np.argmax(np.abs(residual)), residual.shape)
        peak = residual[iy, ix]
        if np.abs(peak) <= threshold:
            break
        model[iy, ix] += gain * peak
        residual -= gain * peak * shift_image(psf, iy - center[0], ix - center[1])
        last_iter = iteration
        if iteration in capture_iters:
            capture(iteration)
    if last_iter not in states:
        capture(last_iter)
    return {'states': states}


def local_rms_map(image, half_window=6):
    n0, n1 = image.shape
    padded = np.pad(image, half_window, mode='reflect')
    rms = np.zeros_like(image)
    for iy in range(n0):
        for ix in range(n1):
            patch = padded[iy:iy + 2 * half_window + 1, ix:ix + 2 * half_window + 1]
            rms[iy, ix] = np.sqrt(np.mean((patch - np.mean(patch)) ** 2))
    return rms


def dilate_mask(mask, radius=4):
    expanded = mask.copy()
    for dy in range(-radius, radius + 1):
        for dx in range(-radius, radius + 1):
            if dy * dy + dx * dx <= radius * radius:
                expanded |= np.roll(np.roll(mask, dy, axis=0), dx, axis=1)
    return expanded


def robust_mad_sigma(values):
    med = np.median(values)
    return 1.4826 * np.median(np.abs(values - med))


def evaluate_clean_progress(dirty, psf, true_sky, support_mask, gain=0.18, niter=2000, clean_beam=None, step=20):
    residual = dirty.copy()
    model = np.zeros_like(dirty)
    center = np.array(psf.shape) // 2
    metrics = {
        'iteration': [0],
        'residual_rms': [np.sqrt(np.mean(residual ** 2))],
        'image_rms': [np.sqrt(np.mean((dirty - true_sky) ** 2))],
        'support_leakage': [0.0],
        'peak_residual': [np.max(np.abs(residual))],
    }

    for iteration in range(1, niter + 1):
        iy, ix = np.unravel_index(np.argmax(np.abs(residual)), residual.shape)
        peak = residual[iy, ix]
        model[iy, ix] += gain * peak
        residual -= gain * peak * shift_image(psf, iy - center[0], ix - center[1])

        if iteration % step == 0:
            model_image = fft_convolve_same(model, clean_beam) if clean_beam is not None else model.copy()
            restored = model_image + residual
            total_model_flux = np.sum(np.abs(model))
            leakage = np.sum(np.abs(model[~support_mask])) / max(total_model_flux, 1e-12)
            metrics['iteration'].append(iteration)
            metrics['residual_rms'].append(np.sqrt(np.mean(residual ** 2)))
            metrics['image_rms'].append(np.sqrt(np.mean((restored - true_sky) ** 2)))
            metrics['support_leakage'].append(leakage)
            metrics['peak_residual'].append(np.max(np.abs(residual)))

    for key in metrics:
        metrics[key] = np.array(metrics[key])
    return metrics


QUALITY_N = 96
UNDER_ITER = 60
NOMINAL_ITER = 560
OVER_ITER = 2000

yy, xx = np.mgrid[0:QUALITY_N, 0:QUALITY_N].astype(float)
trueSky = np.zeros((QUALITY_N, QUALITY_N), dtype=float)
trueSky[25, 24] = 0.90
trueSky[71, 30] = 0.55
trueSky[60, 70] = 0.35
trueSky[28, 76] = 0.22
trueSky += 0.18 * generalGauss2d(48, 48, 4.5, 7.5, amp=1.0, theta=25.0)(xx, yy)

samplingMask, dirtyPSF, uvCoords = make_uv_dirty_beam(QUALITY_N)
cleanBeam = centered_gaussian(QUALITY_N, 2.4, 2.9, theta=18.0)

rng = np.random.default_rng(5)
noiseSigma = 0.05
dirtyImage = fft_convolve_same(trueSky, dirtyPSF) + noiseSigma * rng.standard_normal((QUALITY_N, QUALITY_N))

cleanRun = hogbom_clean(
    dirtyImage,
    dirtyPSF,
    gain=0.18,
    niter=OVER_ITER,
    threshold=0.0,
    clean_beam=cleanBeam,
    capture_iters=(UNDER_ITER, NOMINAL_ITER, OVER_ITER),
)
underState = cleanRun['states'][UNDER_ITER]
nominalState = cleanRun['states'][NOMINAL_ITER]
overState = cleanRun['states'][OVER_ITER]

sourceMask = trueSky > 0.03
supportMask = dilate_mask(sourceMask, radius=4)
progressMetrics = evaluate_clean_progress(
    dirtyImage,
    dirtyPSF,
    trueSky,
    supportMask,
    gain=0.18,
    niter=OVER_ITER,
    clean_beam=cleanBeam,
    step=20,
)


In [ ]:
overLeakage = np.sum(np.abs(overState['model'][~supportMask])) / max(np.sum(np.abs(overState['model'])), 1e-12)
overOutsideComponents = np.count_nonzero(np.abs(overState['model'][~supportMask]) > 1e-6)

print(f'Over-clean iteration count                             = {OVER_ITER}')
print(f'Over-clean residual RMS                                = {np.sqrt(np.mean(overState["residual"] ** 2)):.4f}')
print(f'Model-flux fraction outside true source support        = {overLeakage:.3f}')
print(f'Clean components outside true source support           = {overOutsideComponents}')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
residLimit = 0.16

im0 = axes[0].imshow(overState['residual'], origin='lower', cmap='coolwarm', vmin=-residLimit, vmax=residLimit)
axes[0].set_title(f'Residual image ({OVER_ITER} iterations)')
axes[0].set_xlabel('Pixel')
axes[0].set_ylabel('Pixel')
fig.colorbar(im0, ax=axes[0], shrink=0.78)

im1 = axes[1].imshow(overState['model_image'], origin='lower', cmap='viridis')
axes[1].contour(supportMask.astype(float), levels=[0.5], colors='w', linewidths=0.8)
axes[1].set_title('Beam-convolved CLEAN model')
axes[1].set_xlabel('Pixel')
axes[1].set_ylabel('Pixel')
fig.colorbar(im1, ax=axes[1], shrink=0.78)

plt.tight_layout()


*图：过清理状态下的残图与波束卷积后的 CLEAN 模型。白色轮廓表示真实源的大致支撑区域。残图看起来更“干净”，但模型中已经出现大量落在真实源支撑区域之外的组分，这正是过清理最危险的地方。*


第二个问题是：为什么我们至今仍在大量依赖主观判断来评价图像质量？原因在于，一旦面对真实可见度数据，图像里总会混入各种校准误差。经验丰富的研究者常常能够一眼看出这些误差及其成因，例如增益校准不佳、射频干扰、强源旁瓣或其他系统性问题。错误的校准会导致去卷积发散，从而产生完全不真实的天空模型。人类非常擅长根据图像判断“它看起来是否合理”，却很难把这个过程完整形式化成一个算法。

先看一个最朴素的比较：同一组合成观测的脏图，与接近最佳停止点的复原图。大多数人都会直觉地说后者“更好”，但到底为什么更好、又如何定量说明，正是这一节要解决的问题。


In [ ]:
displayMin = -0.25
displayMax = 0.55

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im0 = axes[0].imshow(dirtyImage, origin='lower', cmap='viridis', vmin=displayMin, vmax=displayMax)
axes[0].set_title('Dirty image')
axes[0].set_xlabel('Pixel')
axes[0].set_ylabel('Pixel')
fig.colorbar(im0, ax=axes[0], shrink=0.78)

im1 = axes[1].imshow(nominalState['restored'], origin='lower', cmap='viridis', vmin=displayMin, vmax=displayMax)
axes[1].set_title(f'Restored image ({NOMINAL_ITER} iterations)')
axes[1].set_xlabel('Pixel')
axes[1].set_ylabel('Pixel')
fig.colorbar(im1, ax=axes[1], shrink=0.78)

plt.tight_layout()


*左：合成观测得到的脏图。右：在接近最佳停止点时得到的复原图。右图中的亮源附近旁瓣明显减弱，扩展结构也更容易辨认。*


与脏图相比，复原图中亮源周围那些明显的 PSF 结构大多已经减弱了。这些结构本质上就是与亮源 PSF 响应相关的成像伪影。去卷积的目标，正是尽可能去掉这些由观测系统引入的结构，用一个更简单、更接近真实天空的模型替代它们。

但这并不意味着“残图越小越好”或者“去卷积越深越好”。下面我们分几种常见指标来看这个问题。


### 6.4.1 动态范围与信噪比


动态范围（dynamic range，DR）是几十年来描述干涉图像质量的经典指标。它通常定义为图像峰值流量 $I_{\textrm{peak}}$ 与图像噪声标准差 $\sigma_I$ 的比值。这个量既可以对脏图计算，也可以对去卷积图计算。


$$\textrm{DR} = \frac{I_{\textrm{peak}}}{\sigma_I}$$


不过，这个定义本身并没有想象中那么明确。首先，“峰值流量”通常被近似成图像中最大的像素值，但像素尺寸和恢复波束都会影响这个值。其次，噪声怎么估计也并不唯一。实践中常见的方案包括：

1. 用整幅复原图的标准差。
2. 用整幅残图的标准差。
3. 用残图的稳健 MAD 估计近似噪声。
4. 在图像角落选一块看起来较空的区域求标准差。
5. 在含有明显源结构的区域求标准差。

下面对同一幅复原图分别采用这些定义，看看动态范围会有多大差异。图中红框表示相对空白的角落区域，青框表示一个含有明显源结构的中心区域。


In [ ]:
cornerSlice = np.s_[0:20, 0:20]
sourceSlice = np.s_[38:58, 38:58]
peakFlux = np.max(nominalState['restored'])

noiseMethods = {
    '整幅复原图标准差': np.std(nominalState['restored']),
    '整幅残图标准差': np.std(nominalState['residual']),
    '残图 MAD 估计': robust_mad_sigma(nominalState['residual']),
    '角落空白区标准差': np.std(nominalState['restored'][cornerSlice]),
    '中心含源区标准差': np.std(nominalState['restored'][sourceSlice]),
}

print(f'Peak flux = {peakFlux:.3f}')
print('Dynamic-range estimates:')
for label, sigma in noiseMethods.items():
    print(f'  {label:<18s} sigma = {sigma:.4f}  DR = {peakFlux / sigma:.2f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im0 = axes[0].imshow(nominalState['restored'], origin='lower', cmap='viridis')
axes[0].add_patch(Rectangle((cornerSlice[1].start, cornerSlice[0].start), cornerSlice[1].stop - cornerSlice[1].start, cornerSlice[0].stop - cornerSlice[0].start, fill=False, edgecolor='tab:red', linewidth=2))
axes[0].add_patch(Rectangle((sourceSlice[1].start, sourceSlice[0].start), sourceSlice[1].stop - sourceSlice[1].start, sourceSlice[0].stop - sourceSlice[0].start, fill=False, edgecolor='cyan', linewidth=2))
axes[0].set_title('Restored image with noise-estimation boxes')
axes[0].set_xlabel('Pixel')
axes[0].set_ylabel('Pixel')
fig.colorbar(im0, ax=axes[0], shrink=0.78)

im1 = axes[1].imshow(nominalState['residual'], origin='lower', cmap='coolwarm', vmin=-0.15, vmax=0.15)
axes[1].set_title('Residual image used for DR estimates')
axes[1].set_xlabel('Pixel')
axes[1].set_ylabel('Pixel')
fig.colorbar(im1, ax=axes[1], shrink=0.78)

plt.tight_layout()


可以看到，同一幅图像只要改变噪声定义，动态范围就会有明显变化。在这个例子里，使用整幅残图、MAD 估计或角落空白区时得到的 DR 大体相近，而如果直接在含有源结构的中心区域估计“噪声”，DR 会显著变小。这说明动态范围并不是一个可以脱离上下文单独引用的数字；在报告 DR 时，必须同时说明噪声是怎样定义的。

更重要的是，动态范围本身只把图像压缩成一个单独数字，无法说明伪影分布在哪里、残图是否相关、模型是否已经开始拟合噪声。因此它有用，但绝不充分。


### 6.4.2 残图


真正有经验的成像工作，往往首先看残图而不是复原图。因为残图直接告诉我们：当前模型还剩下哪些结构没有解释，或者已经解释过头了多少。

下面比较三种状态的残图：

* `60` 次迭代：明显欠清理。
* `560` 次迭代：接近本例的最佳停止点。
* `2000` 次迭代：明显过清理。

每幅残图下面再给出一个局部 RMS 图，用滑动窗口估计各位置的噪声水平。如果残图完全像白噪声，那么局部 RMS 图应该相对平坦；如果残图还带有旁瓣、条纹或大尺度结构，那么这些区域会在 RMS 图上凸显出来。


In [ ]:
stateTable = [
    ('Under-clean', underState),
    ('Near-best stop', nominalState),
    ('Over-clean', overState),
]
residualMaps = [local_rms_map(state['residual'], half_window=6) for _, state in stateTable]

print('Residual diagnostics:')
for (label, state), rmsMap in zip(stateTable, residualMaps):
    leakage = np.sum(np.abs(state['model'][~supportMask])) / max(np.sum(np.abs(state['model'])), 1e-12)
    print(
        f'  {label:<14s} residual RMS = {np.sqrt(np.mean(state["residual"] ** 2)):.4f} | '
        f'local RMS median = {np.median(rmsMap):.4f} | '
        f'local RMS max = {np.max(rmsMap):.4f} | '
        f'support leakage = {leakage:.3f}'
    )

fig, axes = plt.subplots(2, 3, figsize=(16, 10), constrained_layout=True)
residLimit = 0.16
rmsMin = min(np.min(rmsMap) for rmsMap in residualMaps)
rmsMax = max(np.max(rmsMap) for rmsMap in residualMaps)

for col, ((label, state), rmsMap) in enumerate(zip(stateTable, residualMaps)):
    im0 = axes[0, col].imshow(state['residual'], origin='lower', cmap='coolwarm', vmin=-residLimit, vmax=residLimit)
    axes[0, col].set_title(f'{label} residual')
    axes[0, col].set_xlabel('Pixel')
    axes[0, col].set_ylabel('Pixel')

    im1 = axes[1, col].imshow(rmsMap, origin='lower', cmap='magma', vmin=rmsMin, vmax=rmsMax)
    axes[1, col].set_title(f'{label} local RMS')
    axes[1, col].set_xlabel('Pixel')
    axes[1, col].set_ylabel('Pixel')

fig.colorbar(im0, ax=axes[0, :], shrink=0.82)
fig.colorbar(im1, ax=axes[1, :], shrink=0.82)


*图：上排是三种 CLEAN 深度下的残图，下排是对应的局部 RMS 图。欠清理时，残图里仍保留明显结构，局部 RMS 在亮源附近也很不均匀；接近最佳停止点时，残图更接近噪声，RMS 图也更平坦；过清理时，残图继续略微下降，但这并不意味着结果一定更可靠。*


这一组图最值得注意的地方，是“残图更平”与“模型更可信”并不是同一回事。过清理时，残图表面上会继续降低，局部 RMS 也会继续变平，但与此同时，模型里可能已经开始累积大量并不存在于真实天空中的组分。对真实数据而言，我们通常看不到“真实天空”，因此更需要同时检查残图结构、局部 RMS、负碗（negative bowl）、方向相关伪影，以及模型是否稳定。


### 6.4.3 图像质量评价


如何评价干涉图像质量，至今仍然是一个开放问题。如果知道真实天空，例如在模拟里，我们当然可以直接计算保真度指标，例如复原图与真实图之间的 RMS 误差、通量恢复率、虚假源比例等等。但在真实观测中，这些量通常不可得。

因此，真实数据上的图像质量评价往往需要把多种代理指标结合起来看，例如：

* 动态范围。
* 残图 RMS 与峰值残差。
* 局部 RMS 是否均匀。
* 残图中是否仍有明显相关结构。
* 模型对参数变化是否稳定。
* 若回到可见度域，还可以检查残差可见度与 $\chi^2$ 一类统计量。

下面仍用这组合成数据做一个更直接的实验：把 CLEAN 从 0 一直迭代到 2000 次，同时跟踪三条曲线。

* 残图 RMS：它通常会单调下降。
* 复原图相对真实天空的 RMS 误差：只有在模拟里才知道。
* 模型落在真实源支撑区域之外的“泄漏比例”：同样是模拟里才知道，但它非常适合说明过清理的风险。


In [ ]:
bestIndex = np.argmin(progressMetrics['image_rms'])
bestIteration = int(progressMetrics['iteration'][bestIndex])
bestImageRms = progressMetrics['image_rms'][bestIndex]
bestResidualRms = progressMetrics['residual_rms'][bestIndex]
bestLeakage = progressMetrics['support_leakage'][bestIndex]

print(f'Best image-fidelity iteration                = {bestIteration}')
print(f'Image RMS at best-fidelity iteration         = {bestImageRms:.4f}')
print(f'Residual RMS at best-fidelity iteration      = {bestResidualRms:.4f}')
print(f'Support leakage at best-fidelity iteration   = {bestLeakage:.3f}')
print(f'Image RMS after {OVER_ITER} iterations        = {progressMetrics["image_rms"][-1]:.4f}')
print(f'Residual RMS after {OVER_ITER} iterations     = {progressMetrics["residual_rms"][-1]:.4f}')
print(f'Support leakage after {OVER_ITER} iterations  = {progressMetrics["support_leakage"][-1]:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

axes[0].plot(progressMetrics['iteration'], progressMetrics['residual_rms'], label='Residual RMS', linewidth=2)
axes[0].plot(progressMetrics['iteration'], progressMetrics['image_rms'], label='Image RMS to true sky', linewidth=2)
axes[0].axvline(UNDER_ITER, color='tab:orange', linestyle='--', linewidth=1.5)
axes[0].axvline(bestIteration, color='tab:green', linestyle='--', linewidth=1.5)
axes[0].axvline(OVER_ITER, color='tab:red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('CLEAN iteration')
axes[0].set_ylabel('RMS')
axes[0].set_title('Residual RMS versus true-image error')
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(progressMetrics['iteration'], progressMetrics['support_leakage'], color='tab:purple', linewidth=2)
axes[1].axvline(UNDER_ITER, color='tab:orange', linestyle='--', linewidth=1.5)
axes[1].axvline(bestIteration, color='tab:green', linestyle='--', linewidth=1.5)
axes[1].axvline(OVER_ITER, color='tab:red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('CLEAN iteration')
axes[1].set_ylabel('Flux fraction outside true support')
axes[1].set_title('Model leakage outside source support')
axes[1].grid(alpha=0.25)

plt.tight_layout()


这组曲线说明了一个非常关键的事实：在本例中，残图 RMS 几乎一直在下降，但复原图相对真实天空的误差并不是一直下降，而是在大约 `560` 次迭代附近达到最好，之后反而开始变差。与此同时，模型落在真实源支撑区域之外的泄漏比例却持续上升。这正是“过清理”最典型的症状。

因此，真正专业的图像质量评价不应当依赖单一指标。对于模拟，最好同时看保真度、通量恢复、虚假结构和残图统计；对于真实数据，则至少应联合检查动态范围、残图 RMS、局部 RMS、残差结构以及结果对参数的稳定性。`CLEAN` 的停止准则本质上是一个多指标折中问题，而不是一个简单的“残差降到越小越好”的最优化问题。


***

* 下一节： [6.5 源搜索与检测](6_5_source_finding.ipynb)
